# Transcribe Indian-language video with word timestamps

Turn speech into readable text and timed words with **AI4Bharat IndicConformer** and VideoDB Understanding. Start with a short video, explore the result, and export the transcript.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/speech-transcription/indic-conformer.ipynb)

**You will need:** a VideoDB API key with Sandbox Compute access and a short video in your chosen language. Hindi is the starting example. A local file or a video already in your collection works well.

> **Preview:** use an environment with Indic word timestamps enabled. Availability can differ between environments. The default CTC decoder returns words automatically; no timestamp options are needed. Sandbox compute and media processing may incur charges. Stop a sandbox you create when finished.

[Preview the output](#preview) · [Run your video](#connect) · [Explore words](#explore) · [Try more](#try-more) · [Cleanup](#cleanup)

<a id="preview"></a>
## What you get

Each scene contains `data.text` and `data.words`. Word `start` and `end` values are **seconds from the beginning of the video**, even in later scenes. Do not add the scene start again.

The following small example is **illustrative, not a recorded model result**. You can inspect it without an API key. Your video's transcription and timings will differ.

In [1]:
import json

sample_scenes = [
    {"start": 0, "end": 20, "data": {
        "text": "नमस्ते दुनिया", "language": "hi",
        "provider": "open_compute",
        "model": "ai4bharat/indic-conformer-600m-multilingual",
        "timing_precision": "word",
        "words": [
            {"text": "नमस्ते", "start": 0.5, "end": 1.25},
            {"text": "दुनिया", "start": 1.5, "end": 2.0},
        ],
    }},
    {"start": 20, "end": 40, "data": {
        "text": "फिर मिलेंगे", "language": "hi",
        "provider": "open_compute",
        "model": "ai4bharat/indic-conformer-600m-multilingual",
        "timing_precision": "word",
        "words": [
            {"text": "फिर", "start": 20.5, "end": 20.9},
            {"text": "मिलेंगे", "start": 21.0, "end": 21.7},
        ],
    }},
]
print("Illustrative scene response (not a live result):")
print(json.dumps(sample_scenes[1], ensure_ascii=False, indent=2))

Illustrative scene response (not a live result):
{
  "start": 20,
  "end": 40,
  "data": {
    "text": "फिर मिलेंगे",
    "language": "hi",
    "provider": "open_compute",
    "model": "ai4bharat/indic-conformer-600m-multilingual",
    "timing_precision": "word",
    "words": [
      {
        "text": "फिर",
        "start": 20.5,
        "end": 20.9
      },
      {
        "text": "मिलेंगे",
        "start": 21.0,
        "end": 21.7
      }
    ]
  }
}


## 1. Install

Run in Colab or Jupyter. To explore only the sample, run this cell and jump to **Explore the transcript**; no connection or sandbox is needed.

In [ ]:
%pip install -q "videodb>=0.5.1" pandas ipywidgets

<a id="connect"></a>
## 2. Connect

Enter your key securely, or set `VIDEO_DB_API_KEY` in your environment. For a preview environment, set `VIDEODB_API_URL` to the endpoint supplied with your key. Otherwise the SDK uses its standard endpoint.

In [ ]:
import os
from getpass import getpass
from videodb import connect, SandboxTier

api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("VideoDB API key: ")
connection_options = {}
if os.getenv("VIDEODB_API_URL"):
    connection_options["base_url"] = os.environ["VIDEODB_API_URL"]
conn = connect(api_key=api_key, **connection_options)
collection = conn.get_collection()  # Or conn.get_collection("your-collection-id")
print("Connected to VideoDB")

## 3. Choose a video and language

Start with 20–40 seconds of clear speech. Set `VIDEO_SOURCE` to a local path on the notebook machine or a downloadable URL. In Colab, upload your file using the Files panel and use its `/content/...` path.

Alternatively, set `VIDEO_ID` to reuse a video in this collection. Reusing a video avoids another upload. Hindi (`hi`), Tamil (`ta`), Bengali (`bn`), Marathi (`mr`) and Telugu (`te`) are examples of supported languages; choose the language actually spoken.

In [ ]:
MODEL = "ai4bharat/indic-conformer-600m-multilingual"
LANGUAGE = "hi"
VIDEO_ID = ""       # Existing VideoDB video ID, if available.
VIDEO_SOURCE = ""   # Otherwise, e.g. /content/my-hindi-video.mp4
SCENE_SECONDS = 20

if VIDEO_ID:
    video = collection.get_video(VIDEO_ID)
else:
    if not VIDEO_SOURCE:
        raise ValueError("Set VIDEO_SOURCE to your file/URL, or set VIDEO_ID.")
    video = collection.upload(source=VIDEO_SOURCE)
    VIDEO_ID = video.id

print("Video ID:", video.id)
print("Duration:", video.length, "seconds")

## 4. Prepare a sandbox

Both IndicConformer and Whisper use the **small** tier. Set `SANDBOX_ID` to reuse a compatible sandbox you own, or leave it empty to create one for this model. Keep the printed ID if startup takes longer than expected.

Run the creation cell once. If a request times out before an ID is returned, check `conn.list_sandboxes()` before creating another.

In [ ]:
SANDBOX_ID = ""  # Optional: a compatible sandbox already in your account.

if SANDBOX_ID:
    sandbox = conn.get_sandbox(SANDBOX_ID)
else:
    sandbox = conn.create_sandbox(
        tier=SandboxTier.small, models=[MODEL], name="Speech cookbook",
    )
    SANDBOX_ID = sandbox.id
    created_sandbox_id = sandbox.id

print("Sandbox ID:", sandbox.id)

In [ ]:
# Rerun this cell to keep waiting for the same sandbox.
sandbox.wait_for_ready(timeout=2100, interval=15)
if not sandbox.is_active or MODEL not in sandbox.models:
    raise RuntimeError("The selected model is not active in this sandbox. Check it before continuing.")
print("Sandbox ready")

## 5. Transcribe

Name the analyzer `transcript` so it is easy to retrieve. The model, language and sandbox are the only model settings needed here. VideoDB handles audio extraction and model-sized clips; `segmentation` controls how the result is grouped for reading.

Submit once. If polling later times out, rerun the **wait** cell rather than submitting another run. If submission itself times out, inspect `video.list_understandings()` before retrying.

In [ ]:
speech_run = video.understand(
    analyzers=[{
        "type": "spoken_words",
        "name": "transcript",
        "config": {
            "model": MODEL,
            "language": LANGUAGE,
            "sandbox_id": sandbox.id,
        },
    }],
    segmentation={"type": "time", "seconds": SCENE_SECONDS},
)
print("Understanding ID:", speech_run.id)

In [ ]:
speech_run.wait_until_complete(timeout=2700, poll_interval=15)
analyzer = speech_run.get_analyzer("transcript")
if not speech_run.is_successful or not analyzer.is_successful:
    raise RuntimeError(
        f"Transcription did not succeed: run={speech_run.status}, analyzer={analyzer.status}. "
        "Inspect this run before retrying."
    )

transcript_output = analyzer.get_output()
scenes = transcript_output.get("scenes", [])
if not scenes:
    raise RuntimeError("No scenes returned; inspect the analyzer output.")
for scene in scenes:
    data = scene.get("data") or {}
    if data.get("error"):
        raise RuntimeError("A scene contains an error; inspect the analyzer output.")
    if data.get("text", "").strip() and not data.get("words"):
        raise RuntimeError("Text returned without word timings. Check word-timestamp support in this environment.")

print("Transcription complete:", len(scenes), "scenes")

<a id="explore"></a>
## 6. Explore the transcript

Select **sample** to try the explorer without an API call. Select **live** after the transcription above finishes. This section only reads existing results; changing filters does not run the model again.

The scene table shows readable text. The word table shows each word's absolute time. Silent sections may legitimately have no text or words.

In [2]:
import pandas as pd
from IPython.display import display

OUTPUT_SOURCE = "sample"  # Change to "live" after completing your transcription.
if OUTPUT_SOURCE == "live":
    explorer_scenes = scenes
    print("Live output from your transcription")
elif OUTPUT_SOURCE == "sample":
    explorer_scenes = sample_scenes
    print("Illustrative sample only — not your transcription")
else:
    raise ValueError("Choose sample or live.")

scene_table = pd.DataFrame([
    {"start": s["start"], "end": s["end"], "text": s["data"].get("text", ""),
     "words": len(s["data"].get("words", []))}
    for s in explorer_scenes
])
word_table = pd.DataFrame(
    [w for s in explorer_scenes for w in s["data"].get("words", [])],
    columns=["text", "start", "end"],
).drop_duplicates(subset=["text", "start", "end"]).sort_values(["start", "end"])

display(scene_table)
display(word_table.head(20))

Illustrative sample only — not your transcription


,start,end,text,words
0,0,20,नमस्ते दुनिया,2
1,20,40,फिर मिलेंगे,2


,text,start,end
0,नमस्ते,0.5,1.25
1,दुनिया,1.5,2.00
2,फिर,20.5,20.90
3,मिलेंगे,21.0,21.70


### Optional: interactive scene explorer

Choose a scene, search within its words, and toggle its raw JSON. This uses the same scene data shown above. If widgets are unavailable in your notebook viewer, the tables and search cell below work without them.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

# Colab needs its widget manager for interactive controls.
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

scene_picker = widgets.Dropdown(
    options=[(f"{i + 1}: {s['start']:g}–{s['end']:g}s", i)
             for i, s in enumerate(explorer_scenes)],
    description="Scene:",
)
word_filter = widgets.Text(description="Find:", placeholder="Type a word", continuous_update=False)
show_json = widgets.Checkbox(value=False, description="Show JSON")
result_panel = widgets.Output()

def render_scene(change=None):
    scene = explorer_scenes[scene_picker.value]
    data = scene["data"]
    table = pd.DataFrame(data.get("words", []), columns=["text", "start", "end"])
    if word_filter.value.strip():
        table = table[table["text"].str.contains(word_filter.value.strip(), case=False, regex=False, na=False)]
    with result_panel:
        clear_output(wait=True)
        print(data.get("text") or "(No speech in this scene)")
        display(table)
        if show_json.value:
            print(json.dumps(scene, ensure_ascii=False, indent=2))

for control in (scene_picker, word_filter, show_json):
    control.observe(render_scene, names="value")
display(widgets.VBox([widgets.HBox([scene_picker, show_json]), word_filter, result_panel]))
render_scene()

### Find a word across the video

Edit `QUERY` and rerun. This is a local text filter, not a semantic search or another inference call. Use [Indexing](../../indexing/indexing_guide.ipynb) for searchable transcript artifacts.

In [4]:
QUERY = "नमस्ते"
matches = word_table[
    word_table["text"].str.contains(QUERY.strip(), case=False, regex=False, na=False)
] if QUERY.strip() else word_table.iloc[:0]
display(matches)

,text,start,end
0,नमस्ते,0.5,1.25


<a id="try-more"></a>
## Try more

| Experiment | What to change |
|---|---|
| Another Indian language | Set `LANGUAGE` and choose a matching video. Rerun transcription. |
| Compare Whisper | Set `MODEL = "openai/whisper-large-v3-turbo"`, use a sandbox containing that model, then rerun transcription. Both return the same scene/word structure. |
| Larger scene groups | Change `SCENE_SECONDS` from 20 to 30. Word times still refer to the original video. |
| Speaker labels | Use the [managed speech guide](speech-transcription.ipynb). Indic and sandbox Whisper do not provide speaker labels here. |
| Search or summarize speech | Follow [Indexing](../../indexing/indexing_guide.ipynb) or [Multi-analyzer pipelines](../multi-analyzer-pipelines.ipynb). |

Each new transcription creates a new Understanding run. Stop a sandbox you created before switching to another model unless you intend to keep it running.

### Optional: play a matching moment

Only enable this with live results and a matching word. It generates a stream for a small interval around the first match.

In [ ]:
PLAY_MATCH = False
if PLAY_MATCH:
    if OUTPUT_SOURCE != "live" or matches.empty:
        raise ValueError("Select live output and search for a word that exists first.")
    from videodb import play_stream
    match = matches.iloc[0]
    start = max(0, float(match["start"]) - 1)
    end = min(float(video.length), float(match["end"]) + 1)
    display(play_stream(video.generate_stream(timeline=[(start, end)])))

## 7. Export

Save the selected explorer output as JSON and a word-level CSV. Sample exports are named `sample-*` so they cannot be confused with a live transcript. Files are saved on the notebook machine; in Colab, download them from the Files panel.

In [5]:
from pathlib import Path

EXPORT = False
if EXPORT:
    prefix = "sample-transcript" if OUTPUT_SOURCE == "sample" else "transcript"
    Path(f"{prefix}.json").write_text(
        json.dumps({"scenes": explorer_scenes}, ensure_ascii=False, indent=2), encoding="utf-8",
    )
    word_table.to_csv(f"{prefix}-words.csv", index=False)
    print("Saved:", f"{prefix}.json", "and", f"{prefix}-words.csv")

<a id="cleanup"></a>
## 8. Stop your sandbox

Run this when your transcription has finished, including after a failed run. It stops only the sandbox created by this notebook; a reused sandbox is left running. Uploaded videos and completed transcripts remain in your collection.

If creation timed out before returning an ID, inspect `conn.list_sandboxes()` to find the sandbox and stop it when appropriate.

In [ ]:
if globals().get("created_sandbox_id"):
    owned_sandbox = conn.get_sandbox(created_sandbox_id)
    owned_sandbox.stop(grace=True)
    owned_sandbox.wait_for_stop(timeout=300, interval=15)
    print("Sandbox status:", owned_sandbox.status)
else:
    print("No sandbox created by this notebook; nothing to stop.")

### If something needs attention

- **Video URL fails:** upload a local file, or reuse a video already in your collection.
- **Sandbox takes time:** rerun the wait cell using the same sandbox. Cold startup can take several minutes.
- **Understanding fails:** inspect the run and analyzer status. Keep the Understanding ID for support; do not interpret missing words as successful word alignment.
- **Polling times out:** the request may still be running. Continue polling the same run before submitting again.
- **No speech:** check the audio and selected language. Empty words are valid for silence.

See [Outputs and operations](../outputs-and-operations.ipynb) for retrieving an existing run and inspecting failures. Clear private transcript outputs before sharing a notebook.